# 3 — Token + position embeddings

**Before:** notebooks **1–2**.

**This notebook:** `wte` + `wpe` — same idea as GPT-2 / llm.c.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
import torch.nn as nn

text = DATA.read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

block_size = 8
batch_size = 4


In [ ]:
n_embd = 32

token_emb = nn.Embedding(vocab_size, n_embd)
pos_emb = nn.Embedding(block_size, n_embd)

ix = torch.randint(0, len(data) - block_size, (batch_size,))
x = torch.stack([data[i : i + block_size] for i in ix])

tok = token_emb(x)
pos = pos_emb(torch.arange(block_size))
hidden = tok + pos

print("token_emb:", tok.shape)
print("hidden state:", hidden.shape)
